# Five-Class Diabetic Retinopathy Classification

This Google Colab notebook contains the complete workflow: organizing images with `labels.csv`, balancing the classes, preprocessing, training CNN variants, and evaluating them on a shared holdout set. Public outputs are cleared so no retinal images or machine-specific paths are embedded in the repository.

## 1. Colab setup

Upload or extract the Kaggle dataset under `/content/dataset` so it contains `Images/` and `labels.csv`. Change the paths below if the data is stored in Google Drive.

In [ ]:
from pathlib import Path
import random
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    BatchNormalization, Conv2D, Dense, Dropout,
    GlobalAveragePooling2D, Input, MaxPooling2D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATASET_ROOT = Path('/content/dataset')
IMAGES_DIR = DATASET_ROOT / 'Images'
LABELS_CSV = DATASET_ROOT / 'labels.csv'
LABELED_DIR = Path('/content/datasets_split')
BALANCED_DIR = Path('/content/datasets_undersample')
MODEL_DIR = Path('/content/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (180, 180)
TARGET_PER_CLASS = 700
BALANCE_SEED = 42
SPLIT_SEED = 0
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']

## 2. Divide images into folders using `labels.csv`

Each source image is copied to `datasets_split/class_<level>`. This folder name refers to label-based organization, not the later train/validation/test split.

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
required_columns = {'image', 'level'}
if not required_columns.issubset(labels_df.columns):
    raise ValueError(f'labels.csv must contain {sorted(required_columns)}')

if LABELED_DIR.exists():
    shutil.rmtree(LABELED_DIR)
for class_id in range(5):
    (LABELED_DIR / f'class_{class_id}').mkdir(parents=True)

missing_images = []
for row in labels_df.itertuples(index=False):
    source = IMAGES_DIR / f'{row.image}.jpeg'
    if not source.exists():
        missing_images.append(source.name)
        continue
    destination = LABELED_DIR / f'class_{int(row.level)}' / source.name
    shutil.copy2(source, destination)

class_counts = {
    class_id: len(list((LABELED_DIR / f'class_{class_id}').glob('*.jpeg')))
    for class_id in range(5)
}
print('Class counts:', class_counts)
print('Missing images:', len(missing_images))

## 3. Balance the dataset

The original distribution is 25,802 / 2,438 / 5,288 / 872 / 708. Seed-42 undersampling keeps 700 images from each class, producing 3,500 balanced images.

In [ ]:
if BALANCED_DIR.exists():
    shutil.rmtree(BALANCED_DIR)
BALANCED_DIR.mkdir(parents=True)

rng = random.Random(BALANCE_SEED)
for class_id in range(5):
    source_dir = LABELED_DIR / f'class_{class_id}'
    destination_dir = BALANCED_DIR / f'class_{class_id}'
    destination_dir.mkdir()
    image_files = sorted(source_dir.glob('*.jpeg'))
    if len(image_files) < TARGET_PER_CLASS:
        raise ValueError(f'class_{class_id} has only {len(image_files)} images')
    for source in rng.sample(image_files, TARGET_PER_CLASS):
        shutil.copy2(source, destination_dir / source.name)

print('Balanced images:', sum(1 for _ in BALANCED_DIR.glob('class_*/*.jpeg')))

## 4. Load, preprocess, and split

Images are read with OpenCV, converted from BGR to RGB, resized to 180×180, cast to `float32`, and normalized to `[0, 1]`. The stratified split is 70% training, 15% validation, and 15% testing.

In [ ]:
images, labels = [], []
for class_id in range(5):
    for image_path in sorted((BALANCED_DIR / f'class_{class_id}').glob('*.jpeg')):
        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
        images.append(image.astype('float32'))
        labels.append(class_id)

X = np.asarray(images, dtype=np.float32) / 255.0
y = np.asarray(labels, dtype=np.int64)

X_train, X_other, y_train, y_other = train_test_split(
    X, y, test_size=0.30, random_state=SPLIT_SEED, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_other, y_other, test_size=0.50, random_state=SPLIT_SEED, stratify=y_other
)
print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_valid.shape, y_valid.shape)
print('Test:', X_test.shape, y_test.shape)

## 5. CNN and evaluation helpers

In [ ]:
def build_cnn(dense_units, dropout_rate):
    model = Sequential([
        Input(shape=(*IMAGE_SIZE, 3)),
        Conv2D(32, 3, padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(32, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(),
        Conv2D(64, 3, padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(64, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(),
        Conv2D(128, 3, padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(128, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(),
        GlobalAveragePooling2D(),
        Dense(dense_units, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(5, activation='softmax'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

def evaluate_model(model):
    predictions = np.argmax(model.predict(X_test, verbose=0), axis=1)
    scores = {
        'accuracy': accuracy_score(y_test, predictions),
        'macro_precision': precision_score(y_test, predictions, average='macro', zero_division=0),
        'macro_recall': recall_score(y_test, predictions, average='macro', zero_division=0),
        'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
    }
    print(scores)
    print(classification_report(y_test, predictions, target_names=CLASS_NAMES, zero_division=0))
    print(confusion_matrix(y_test, predictions))
    return scores

## 6. Baseline CNN

In [ ]:
baseline_model = build_cnn(dense_units=128, dropout_rate=0.4)
baseline_history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=10, batch_size=32,
)
baseline_scores = evaluate_model(baseline_model)

## 7. Tuned CNN with early stopping

In [ ]:
tuned_model = build_cnn(dense_units=125, dropout_rate=0.3)
early_stopping = EarlyStopping(
    monitor='val_accuracy', patience=8, restore_best_weights=True
)
tuned_history = tuned_model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=50, batch_size=32,
    callbacks=[early_stopping],
)
tuned_scores = evaluate_model(tuned_model)
tuned_model.save(MODEL_DIR / 'CNNv2.keras')

## 8. Exploratory augmentation continuation

This block continues training the tuned model with augmentation. It is an exploratory fine-tuning stage, not an independently initialized model comparison.

In [ ]:
augmenter = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    brightness_range=[0.9, 1.1],
    horizontal_flip=True,
    fill_mode='nearest',
)
augmentation_history = tuned_model.fit(
    augmenter.flow(X_train, y_train, batch_size=32),
    validation_data=(X_valid, y_valid),
    epochs=50, callbacks=[early_stopping],
)
augmentation_scores = evaluate_model(tuned_model)

## Recorded Colab results

| Experiment | Best validation accuracy | Test accuracy | Macro precision | Macro recall | Macro F1 |
|---|---:|---:|---:|---:|---:|
| Baseline CNN | 33.33% | 26.29% | 15.36% | 26.29% | 18.36% |
| Tuned CNN with early stopping | **39.05%** | **38.48%** | **42.38%** | **38.48%** | **33.94%** |
| Continued training with augmentation | 21.71% | 21.52% | 8.82% | 21.52% | 11.80% |

These values are transcribed from the stored outputs of the original Colab notebook and were not rerun in this public repository. The tuned run reached its best validation accuracy at epoch 20. Values may differ when re-running because the original experiment did not fix every TensorFlow/NumPy source of randomness.